In [ ]:
import pandas as pd

chemin_no2 = "../../data/raw/geodair/no2_paris_a1_01_2015_11_2026.csv"
apercu = pd.read_csv(chemin_no2, sep=";", nrows=5)
apercu

In [ ]:
apercu.iloc[0]

In [ ]:
colonnes = ["Date de début", "code site", "nom site", "type d'influence", "valeur", "validité"]
no2 = pd.read_csv(chemin_no2, sep=";", usecols=colonnes)
no2.shape

In [ ]:
no2.groupby(["code site", "nom site", "type d'influence"]).size()

In [ ]:
chemin_pm25 = "../../data/raw/geodair/pm25_paris_a1_01_2015_11_2026.csv"
pm25 = pd.read_csv(chemin_pm25, sep=";", usecols=colonnes)
pm25.groupby(["code site", "nom site", "type d'influence"]).size()

In [ ]:
pm25.groupby("nom site")["Date de début"].agg(["min", "max"])

In [ ]:
fond_pm25 = pm25[pm25["type d'influence"] == "Fond"]
depuis_2022 = fond_pm25[fond_pm25["Date de début"] >= "2022-01-01"]
comparaison = depuis_2022.pivot_table(index="Date de début", columns="nom site", values="valeur")
comparaison.describe()

In [ ]:
# Panne de capteur, pour enlever les mesures prises par des capteurs défectueux
# 1 mesure correcte, -1 capteurs en panne
print(no2["validité"].value_counts())
print(pm25["validité"].value_counts())

In [ ]:
# Fonction pour garder que les mesures valides
def nettoyer(df, stations):
    df = df[df["code site"].isin(stations)]
    df = df[(df["validité"] == 1) & (df["valeur"] >= 0)]
    df = df.assign(date=pd.to_datetime(df["Date de début"]))
    return df.groupby("date")["valeur"].mean()

In [ ]:
stations_no2 = ["FR04004", "FR04014", "FR04037", "FR04060"]
stations_pm25 = ["FR04004", "FR04055", "FR04143"]

no2_horaire = nettoyer(no2, stations_no2)
pm25_horaire = nettoyer(pm25, stations_pm25)

print(no2_horaire.shape, pm25_horaire.shape)

In [ ]:
no2_jour = no2_horaire.resample("D").mean()
pm25_jour = pm25_horaire.resample("D").mean()

print(no2_jour.isna().sum(), pm25_jour.isna().sum())

In [ ]:
pm25_jour.plot(figsize=(15, 4), title="PM2.5 - moyenne journalière, stations de fond Paris")

In [ ]:
# Nombre de jours sans aucune mesure par années
# 2019 a bcp de valeur manquante car il s'agit d'une année avec chg de station
pm25_jour[pm25_jour.isna()].index.year.value_counts().sort_index()

In [ ]:
no2_jour.plot(figsize=(15, 4), title="NO2 - moyenne journalière, stations de fond Paris")


In [ ]:
from prophet import Prophet

df_no2 = no2_jour.reset_index()
df_no2.columns = ["ds", "y"]

train = df_no2[df_no2["ds"] < "2025-01-01"] # train 2016 - 2024
test = df_no2[df_no2["ds"] >= "2025-01-01"] # test 2024 - 2026

modele = Prophet()
modele.fit(train)
prevision = modele.predict(test[["ds"]])

In [ ]:
import numpy as np

# erreur moyenne
mae_prophet = np.abs(test["y"].values - prevision["yhat"].values).mean()

# Baseline persistance, si la prédiction de j+1 reprend la valeur de la veille
hier = df_no2["y"].shift(1)[df_no2["ds"] >= "2025-01-01"]
mae_persistance = np.abs(test["y"].values - hier.values).mean()

print("Prophet :", round(mae_prophet, 2), "| Persistance :", round(mae_persistance, 2))
print("Donc prophet fait moins bien que la baseline")

In [ ]:
# Prophet NO2
modele.plot_components(prevision);

In [ ]:
#
def evaluer(serie):
    df = serie.reset_index()
    df.columns = ["ds", "y"]
    train = df[df["ds"] < "2025-01-01"]
    test = df[df["ds"] >= "2025-01-01"]

    modele = Prophet().fit(train)
    prevision = modele.predict(test[["ds"]])

    mae_prophet = np.nanmean(np.abs(test["y"].values - prevision["yhat"].values))
    hier = df["y"].shift(1)[df["ds"] >= "2025-01-01"]
    mae_persistance = np.nanmean(np.abs(test["y"].values - hier.values))

    print("Prophet :", round(mae_prophet, 2), "| Persistance :", round(mae_persistance, 2))
    return modele, prevision

In [ ]:
modele_pm25, prevision_pm25 = evaluer(pm25_jour)

# Prophet PM2.5
modele_pm25.plot_components(prevision_pm25);

## Conclusions
- Stations :
  - NO2 : prise des moyenne des stations de fond (18e, 12e, 13e, 7e)
  - PM2.5 :
    - 2015 à sept 2019: Paris Centre
    - oct 2019 à 2022 : Les Halles
    - 2022 à 2024 : Les Halles + 18e
- Données de test: 2024 à 2026 sur les stations Les Halles + 18e
- Nettoyage :
  - On a gardé que les valeurs valide (validité = 1),
  - On a recalculé les moyenne journalières à partir des moyennes horaires
- Split temporel : train 2015-2024, test 2025-2026
- MAE NO2 : Prophet 6.25 vs persistance 5.71 (écart de 0.50)
- MAE PM2.5 : Prophet 4.68 vs persistance 3.13 (écart 1.8)
- ! Prophet est moins bon que la baseline persistance
- On constate également que le calendrier explique bien le NO2 (majorité traffic, diesel), mal les PM2.5 (traffic usure pneus+frein, chauffage bois/fuel, agriculture et méto/vent)
  - Hypothèse: la météo peut améliorer les prédiction pour les PM2.5
  - Le NO2


In [ ]:
# Mesure de corrélation entre les stations de fonds et les stations du périphérique
# Plus la corrélation est élevée, plus les taux mesurés sont proches, donc moins le traffic aurait d'influence

trafic_pm25 = nettoyer(pm25, ["FR04329"]).resample("D").mean()
trafic_no2 = nettoyer(no2, ["FR04329"]).resample("D").mean()

print("PM2.5 | corrélation :", round(pm25_jour.corr(trafic_pm25), 2), "| fond :", round(pm25_jour.mean(), 1), "| trafic :", round(trafic_pm25.mean(), 1))

print("NO2   | corrélation :", round(no2_jour.corr(trafic_no2), 2), "| fond :", round(no2_jour.mean(), 1), "| trafic :", round(trafic_no2.mean(), 1))

## Test propagation : fond vs trafic (périph' Est FR04329)
| | Corrélation fond/trafic | Fond | Trafic | Surplus trafic |
|---|---|---|---|---|
| PM2.5 | 0.93 | 12.1 | 14.7 | +21 % |
| NO2 | 0.74 | 27.3 | 54.0 | x2 |


- PM2.5 : les taux mesurés sur les stations de fond et de trafic sont quasi identiques
  - comportement de pollution régionale, le trafic n'ajoute qu'un petit surplus local

- NO2 : les taux mesurés sur les stations de traffic est 2 fois plus élevé que les stations de fond
  - pollution locale

- Limite : une forte corrélation peut aussi venir de la météo commune aux deux stations. Pour prouver la propagation, il faudra des stations rurales et la direction du vent

- Conséquence pour le modèle : les PM2.5 dépendent de phénomènes à grande échelle (météo, masses d'air)
  - features météo prioritaires, dont la direction du vent